# 1、字符串输出解析器 StrOutputParser

In [2]:
# 1、获取大模型
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser, XMLOutputParser

import os
import dotenv
from langchain_core.utils import pre_init
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini")

# 2、调用大模型
response = chat_model.invoke("什么是大语言模型？")
# print(type(response))   #AIMessage

#3、如何获取一个字符串的输出结果呢？
# 方式1：自己调用输出结果的content
# print(response.content)

# 方式2：使用StrOutputParser
parser = StrOutputParser()
str_response = parser.invoke(response)
print(type(str_response))  #<class 'str'>
print(str_response)


<class 'str'>
大语言模型（Large Language Model, LLM）是一种基于深度学习的自然语言处理（NLP）模型，它能够理解和生成自然语言文本。这类模型通常使用大量的文本数据进行训练，能够捕捉语言的结构、语法、语义及上下文信息。

大语言模型的特点包括：

1. **规模庞大**：模型参数通常在亿级以上，甚至达到万亿级，因而具备很强的学习能力和表达能力。
  
2. **预训练与微调**：通常采用预训练和微调的方式，先在大规模通用语料上进行无监督预训练，然后在特定任务或领域的数据上进行有监督的微调。

3. **多任务能力**：可以在多种自然语言处理任务上表现良好，包括文本生成、翻译、问答、摘要等。

4. **上下文理解**：通过自注意力机制，能够有效捕捉长距离依赖关系，从而更好地理解上下文。

大语言模型的应用广泛，包括聊天机器人、文本生成、内容创作、语言翻译等。随着技术的进步，它们在智能助手和自动化内容生成中的应用也日益增多。


# 2、JsonOutputParser : Json输出解析器

方式1：

In [3]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

chat_model = ChatOpenAI(model="gpt-4o-mini")

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个靠谱的{role}"),
    ("human", "{question}")
])

# 正确的：
prompt = chat_prompt_template.invoke(
    input={"role": "人工智能专家", "question": "人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式的数据"})

# 错误的：错误原因：因为没有加提示词限制，那么大模型返回的结果不会天然的具有Json格式，所以无法转成json，但是如果加了限制，那么返回的结果虽然格式仍然是字符串，但是已经具有了json的形式。
# prompt = chat_prompt_template.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说？"})

response = chat_model.invoke(prompt)
# print(response.content)

# 获取一个JsonOutputParser的实例
parser = JsonOutputParser()

json_result = parser.invoke(response)
print(json_result)

{'q': '人工智能用英文怎么说？', 'a': 'Artificial Intelligence'}


方式2：

举例1：

In [9]:
parser = JsonOutputParser()

print(parser.get_format_instructions())

Return a JSON object.


举例2：

In [4]:
# 引入依赖包
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

joke_query = "告诉我一个笑话。"

# 定义Json解析器
parser = JsonOutputParser()

#以PromptTemplate为例
prompt_template = PromptTemplate.from_template(
    template="回答用户的查询\n 满足的格式为{format_instructions}\n 问题为{question}\n",
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

prompt = prompt_template.invoke(input={"question": joke_query})
response = chat_model.invoke(prompt)
print(response)

json_result = parser.invoke(response)
print(json_result)

content='```json\n{\n  "joke": "为什么书总是很冷？因为它们有很多的封面！"\n}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 32, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D5m3F9w1I5B80qOztSNENLjTDAW8b', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--43d88ca2-8350-4f39-8895-b58ef3393586-0' usage_metadata={'input_tokens': 32, 'output_tokens': 30, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
{'joke': '为什么书总是很冷？因为它们有很多的封面！'}


知识的拓展： |

In [5]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

chat_model = ChatOpenAI(model="gpt-4o-mini")

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个靠谱的{role}"),
    ("human", "{question}")
])

# 获取一个JsonOutputParser的实例
parser = JsonOutputParser()

# 写法1：
# prompt = chat_prompt_template.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式的数据"})
#
# response = chat_model.invoke(prompt)
#
# json_result = parser.invoke(response)
# print(json_result)


# 写法2：定义了一个管道，输入是第一步最开始的输入，直接输出最终结果
chain = chat_prompt_template | chat_model | parser
json_result1 = chain.invoke(
    input={"role": "人工智能专家", "question": "人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式的数据"})
print(json_result1)

{'q': '人工智能用英文怎么说？', 'a': 'Artificial Intelligence'}


针对于举例2

In [7]:
# 引入依赖包
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

joke_query = "告诉我一个笑话。"

# 定义Json解析器
parser = JsonOutputParser()

#以PromptTemplate为例
prompt_template = PromptTemplate.from_template(
    template="回答用户的查询\n 满足的格式为{format_instructions}\n 问题为{question}\n",
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
# 写法1：
# prompt = prompt_template.invoke(input={"question":joke_query})
# response = chat_model.invoke(prompt)
# json_result = parser.invoke(response)

chain = prompt_template | chat_model | parser
json_result = chain.invoke(input={"question": joke_query})
print(json_result)

{'joke': '为什么鸡过马路？为了到另一边！'}


# 3、XMLOutputParser XML输出解析器的使用

举例1：自己在提示词模板中写明使用XML格式

In [8]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

actor_query = "周星驰的简短电影记录"
response = chat_model.invoke(f"请生成{actor_query}，将影片附在<movie></movie>标签中")

print(type(response))
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
周星驰（Stephen Chow）是中国著名的电影演员、导演和编剧，以其独特的喜剧风格和幽默感而闻名。他的电影往往结合了搞笑、情感和对人生的思考，深受观众喜爱。以下是一些周星驰经典电影的简短记录：

<movie>
    <title>大话西游之月光宝盒</title>
    <year>1995</year>
    <summary>讲述了至尊宝与紫霞仙子的爱情故事，以及他在时空旅行中经历的种种奇遇。这部电影融合了喜剧与哲理，影响深远。</summary>
</movie>

<movie>
    <title>功夫</title>
    <year>2004</year>
    <summary>一部以武侠为主题的喜剧片，讲述了一个小混混在机缘巧合下，学会功夫并与黑帮势力展开斗争的故事。这部电影赢得了广泛的赞誉和多个奖项。</summary>
</movie>

<movie>
    <title>食神</title>
    <year>1996</year>
    <summary>周星驰饰演的主角是一个被逐出厨师界的“食神”，在重返烹饪界的过程中，他经历了种种挫折并最终找到了自己的烹饪灵感。</summary>
</movie>

<movie>
    <title>逃学威龙</title>
    <year>1991</year>
    <summary>讲述了一位警察假扮学生混入学校调查犯罪的搞笑故事。影片充满了幽默与社会讽刺。</summary>
</movie>

<movie>
    <title>喜剧之王</title>
    <year>1999</year>
    <summary>周星驰饰演的梦想成为演员的男人，通过不懈努力迎来了自己的成功，同时也传递了对梦想和爱情的执着追求。</summary>
</movie>

这些电影不仅展现了周星驰的喜剧才华，也反映了社会的多种现象和人性的深刻思考。


举例2：

In [17]:
from langchain_core.output_parsers.xml import XMLOutputParser

parser = XMLOutputParser()
print(parser.get_format_instructions())

The output should be formatted as a XML file.
1. Output should conform to the tags below.
2. If tags are not given, make them on your own.
3. Remember to always open and close all the tags.

As an example, for the tags ["foo", "bar", "baz"]:
1. String "<foo>
   <bar>
      <baz></baz>
   </bar>
</foo>" is a well-formatted instance of the schema.
2. String "<foo>
   <bar>
   </foo>" is a badly-formatted instance.
3. String "<foo>
   <tag>
   </tag>
</foo>" is a badly-formatted instance.

Here are the output tags:
```
None
```


使用parser.get_format_instructions()结构实现：

In [9]:
# 1.导入相关包
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 2. 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 3.测试模型的xml解析效果
actor_query = "生成汤姆·汉克斯的简短电影记录,使用中文回复"

# 4.定义XMLOutputParser对象
parser = XMLOutputParser()

# 5. 生成提示词模板
prompt_template1 = PromptTemplate.from_template(
    template="用户的问题：{query}\n使用的格式：{format_instructions}"
)

prompt_template2 = prompt_template1.partial(format_instructions=parser.get_format_instructions())


response = chat_model.invoke(prompt_template2.invoke(input={"query": actor_query}))
print(response.content)


```xml
<电影记录>
   <演员>
      <名字>汤姆·汉克斯</名字>
      <出生日期>1956-07-09</出生日期>
      <国籍>美国</国籍>
   </演员>
   <电影列表>
      <电影>
         <标题>阿甘正传</标题>
         <年份>1994</年份>
         <角色>阿甘</角色>
         <类型>剧情/喜剧</类型>
         <获奖情况>获得2项奥斯卡奖</获奖情况>
      </电影>
      <电影>
         <标题>拯救大兵瑞恩</标题>
         <年份>1998</年份>
         <角色>米勒上尉</角色>
         <类型>战争/剧情</类型>
         <获奖情况>获得5项奥斯卡奖</获奖情况>
      </电影>
      <电影>
         <标题>时报广场的情侣</标题>
         <年份>1990</年份>
         <角色>乔治</角色>
         <类型> романтика</类型>
         <获奖情况>无</获奖情况>
      </电影>
   </电影列表>
</电影记录>
```


In [12]:
xml_result = parser.invoke(response)
print(xml_result)

{'电影记录': [{'演员': [{'名字': '汤姆·汉克斯'}, {'出生日期': '1956-07-09'}, {'国籍': '美国'}]}, {'电影列表': [{'电影': [{'标题': '阿甘正传'}, {'年份': '1994'}, {'角色': '阿甘'}, {'类型': '剧情/喜剧'}, {'获奖情况': '获得2项奥斯卡奖'}]}, {'电影': [{'标题': '拯救大兵瑞恩'}, {'年份': '1998'}, {'角色': '米勒上尉'}, {'类型': '战争/剧情'}, {'获奖情况': '获得5项奥斯卡奖'}]}, {'电影': [{'标题': '时报广场的情侣'}, {'年份': '1990'}, {'角色': '乔治'}, {'类型': ' романтика'}, {'获奖情况': '无'}]}]}]}


# 4、列表解析器 CommaSeparatedListOutputParser

举例1：

In [23]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

messages = "大象,猩猩,狮子"
result = output_parser.parse(messages)
print(result)
print(type(result))

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['大象', '猩猩', '狮子']
<class 'list'>


举例2：

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers import CommaSeparatedListOutputParser

# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 创建解析器
output_parser = CommaSeparatedListOutputParser()

# 创建LangChain提示模板
chat_prompt = PromptTemplate.from_template(
    "生成5个关于{text}的列表.\n\n{format_instructions}",
    partial_variables={
    "format_instructions": output_parser.get_format_instructions()
    })

# 提示模板与输出解析器传递输出
# chat_prompt = chat_prompt.partial(format_instructions=output_parser.get_format_instructions())

# 将提示和模型合并以进行调用
chain = chat_prompt | chat_model | output_parser
res = chain.invoke({"text": "电影"})
print(res)
print(type(res))

['1. 经典电影: 《公民凯恩》，《教父》，《 casablanca》，《惊魂记》，《2001太空漫游》', '2. 热门科幻电影: 《星际穿越》，《盗梦空间》，《银翼杀手2049》，《黑客帝国》，《火星救援》', '3. 经典动画电影: 《狮子王》，《冰雪奇缘》，《千与千寻》，《沃尔-E》，《美丽心灵》', '4. 电影导演: 斯皮尔伯格，希区柯克，塔伦蒂诺，诺兰，库布里克', '5. 著名电影配乐: 《星球大战》，《哈利·波特》，《盗梦空间》，《教父》，《指环王》']
<class 'list'>


# 5、日期解析器 DatetimeOutputParser

举例1：

In [25]:
from langchain.output_parsers import DatetimeOutputParser

output_parser = DatetimeOutputParser()

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.

Examples: 1853-04-07T06:45:24.945854Z, 0835-04-16T12:35:51.978680Z, 1877-10-31T12:04:45.397316Z

Return ONLY this string, no other words!


举例2：


In [14]:
from langchain_openai import ChatOpenAI
from langchain.prompts.chat import HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain.output_parsers import DatetimeOutputParser

chat_model = ChatOpenAI(model="gpt-4o-mini")


chat_prompt = ChatPromptTemplate.from_messages([
    ("system","{format_instructions}"),
    ("human", "{request}")
])

output_parser = DatetimeOutputParser()

# 方式1：
# model_request = chat_prompt.format_messages(
#     request="中华人民共和国是什么时候成立的",
#     format_instructions=output_parser.get_format_instructions()
# )

# response = chat_model.invoke(model_request)
# result = output_parser.invoke(response)
# print(result)
# print(type(result))

# 方式2：
chain = chat_prompt | chat_model | output_parser
resp = chain.invoke({"request":"中华人民共和国是什么时候成立的",
                     "format_instructions":output_parser.get_format_instructions()})
print(resp)
print(type(resp))

1949-10-01 00:00:00
<class 'datetime.datetime'>
